In [ ]:
from google.colab import userdata
import os

# Access your secrets
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')
os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')

# Now you can download directly without a kaggle.json file
!kaggle datasets download -d andrewmvd/ocular-disease-recognition-odir5k

Dataset URL: https://www.kaggle.com/datasets/andrewmvd/ocular-disease-recognition-odir5k
License(s): other
 99% 1.60G/1.62G [00:09<00:00, 244MB/s]
100% 1.62G/1.62G [00:09<00:00, 188MB/s]


In [ ]:
# Unzip the downloaded file
import zipfile
import os

zip_ref = zipfile.ZipFile('ocular-disease-recognition-odir5k.zip', 'r')
zip_ref.extractall('odir_data') # This extracts everything into a folder named 'odir_data'
zip_ref.close()

print("Dataset is ready!")

Dataset is ready!


In [ ]:
# View the contents
os.listdir('odir_data')

['preprocessed_images', 'full_df.csv', 'ODIR-5K']

In [ ]:
import pandas as pd
import numpy as np
import cv2
import os
import tensorflow as tf
from tensorflow.keras import layers, models, Input
from tensorflow.keras.applications import EfficientNetB3
from sklearn.model_selection import train_test_split
from google.colab import drive

# --- 0. CONNECT GOOGLE DRIVE (Safety First) ---
drive.mount('/content/drive')
SAVE_PATH = '/content/drive/MyDrive/best_siamese_model.keras'

# --- 1. PREPROCESSING ---
def preprocess_fundus(image_path):
    if not os.path.exists(image_path):
        return np.zeros((512, 512, 3), dtype='float32')

    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (512, 512))

    # Ben Graham's method: Highlights lesions and vessels
    image = cv2.addWeighted(image, 4, cv2.GaussianBlur(image, (0,0), 10), -4, 128)
    return image.astype('float32') / 255.0

# --- 2. DATA LOADING ---
target_cols = ['N', 'D', 'G', 'C', 'A', 'H', 'M', 'O']
full_df = pd.read_csv('odir_data/full_df.csv')
img_dir = "odir_data/preprocessed_images"

train_df, val_df = train_test_split(full_df, test_size=0.2, random_state=42)

# --- 3. SIAMESE GENERATOR ---
def siamese_generator(dataframe, batch_size=8):
    while True:
        df = dataframe.sample(frac=1).reset_index(drop=True)
        for i in range(0, len(df), batch_size):
            batch_df = df.iloc[i:i+batch_size]
            l_batch, r_batch, y_batch = [], [], []

            for _, row in batch_df.iterrows():
                l_img = preprocess_fundus(f"{img_dir}/{row['Left-Fundus']}")
                r_img = preprocess_fundus(f"{img_dir}/{row['Right-Fundus']}")
                l_batch.append(l_img)
                r_batch.append(r_img)
                y_batch.append(row[target_cols].values.astype('float32'))

            yield (np.array(l_batch, dtype='float32'), np.array(r_batch, dtype='float32')), np.array(y_batch, dtype='float32')

# --- 4. LOSS & ARCHITECTURE ---
def focal_loss(gamma=2.0, alpha=0.25):
    def focal_loss_fixed(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        bce = tf.keras.backend.binary_crossentropy(y_true, y_pred)
        p_t = (y_true * y_pred) + ((1 - y_true) * (1 - y_pred))
        alpha_factor = y_true * alpha + (1 - y_true) * (1 - alpha)
        modulating_factor = tf.pow(1.0 - p_t, gamma)
        return tf.reduce_mean(alpha_factor * modulating_factor * bce)
    return focal_loss_fixed

# Siamese Build
base_model = EfficientNetB3(weights='imagenet', include_top=False, input_shape=(512, 512, 3))
base_model.trainable = True

left_input = Input(shape=(512, 512, 3), name="left_eye")
right_input = Input(shape=(512, 512, 3), name="right_eye")

left_feats = layers.GlobalAveragePooling2D()(base_model(left_input))
right_feats = layers.GlobalAveragePooling2D()(base_model(right_input))

combined = layers.Concatenate()([left_feats, right_feats])
x = layers.Dense(512, activation='relu')(combined)
x = layers.Dropout(0.4)(x)
output = layers.Dense(8, activation='sigmoid')(x)

model = models.Model(inputs=[left_input, right_input], outputs=output)

auc_metric = tf.keras.metrics.AUC(multi_label=True, name='auc_performance')
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5), loss=focal_loss(), metrics=['accuracy', auc_metric])

# --- 5. DATASET WRAPPING ---
output_signature = (
    (tf.TensorSpec(shape=(None, 512, 512, 3), dtype=tf.float32), tf.TensorSpec(shape=(None, 512, 512, 3), dtype=tf.float32)),
    tf.TensorSpec(shape=(None, 8), dtype=tf.float32)
)

train_dataset = tf.data.Dataset.from_generator(lambda: siamese_generator(train_df, 8), output_signature=output_signature).prefetch(8)
val_dataset = tf.data.Dataset.from_generator(lambda: siamese_generator(val_df, 8), output_signature=output_signature).prefetch(8)

# --- 6. CALLBACKS & TRAINING ---
# Monitor val_accuracy this time since you want > 90%
checkpoint = tf.keras.callbacks.ModelCheckpoint(SAVE_PATH, monitor='val_auc_performance', save_best_only=True, mode='max', verbose=1)
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_auc_performance', patience=7, restore_best_weights=True)

print("Starting Training... Model will be saved to your Google Drive automatically.")
history = model.fit(
    train_dataset,
    steps_per_epoch=len(train_df)//8,
    validation_data=val_dataset,
    validation_steps=len(val_df)//8,
    epochs=40,
    callbacks=[checkpoint, early_stop]
)

# Final Save
model.save("/content/drive/MyDrive/final_siamese_ocular_model.keras")
print("All done! Check your Google Drive for the model file.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Starting Training... Model will be saved to your Google Drive automatically.
Epoch 1/40
639/639 ━━━━━━━━━━━━━━━━━━━━ 0s 608ms/step - accuracy: 0.2825 - auc_performance: 0.5439 - loss: 0.0614
Epoch 1: val_auc_performance improved from -inf to 0.73102, saving model to /content/drive/MyDrive/best_siamese_model.keras
639/639 ━━━━━━━━━━━━━━━━━━━━ 710s 761ms/step - accuracy: 0.2826 - auc_performance: 0.5440 - loss: 0.0614 - val_accuracy: 0.4701 - val_auc_performance: 0.7310 - val_loss: 0.0320
Epoch 2/40
639/639 ━━━━━━━━━━━━━━━━━━━━ 0s 622ms/step - accuracy: 0.3988 - auc_performance: 0.6872 - loss: 0.0330
Epoch 2: val_auc_performance improved from 0.73102 to 0.74134, saving model to /content/drive/MyDrive/best_siamese_model.keras
639/639 ━━━━━━━━━━━━━━━━━━━━ 578s 739ms/step - accuracy: 0.3988 - auc_performance: 0.6873 - loss: 0.0330 - val_accuracy: 0.4347 - val_auc_

In [ ]:
# 4. CHECK PARAMETERS HERE!
print(f"Total parameters: {model.count_params()}")
# This is even better because it breaks them down:
model.summary()

Total parameters: 10801975


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb3 (Functional)     │ (None, 10, 10, 1536)   │    10,783,535 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1536)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1536)           │         6,144 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1536)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 8)              │        12,296 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,801,975 (41.21 MB)

 Trainable params: 10,711,600 (40.86 MB)

 Non-trainable params: 90,375 (353.03 KB)

In [ ]:
import matplotlib.pyplot as plt

def plot_training_results(history):
    # Set up the figure
    plt.figure(figsize=(14, 5))

    # Plot 1: AUC (Area Under Curve) - Your "North Star" metric
    plt.subplot(1, 2, 1)
    plt.plot(history.history['auc_performance'], label='Training AUC', color='#1f77b4', linewidth=2)
    plt.plot(history.history['val_auc_performance'], label='Validation AUC', color='#ff7f0e', linewidth=2)
    plt.title('Model AUC (Diagnostic Quality)', fontsize=14)
    plt.xlabel('Epochs')
    plt.ylabel('AUC Score')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()

    # Plot 2: Focal Loss - Shows how well the model handled rare diseases
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Training Loss', color='#d62728', linewidth=2)
    plt.plot(history.history['val_loss'], label='Validation Loss', color='#2ca02c', linewidth=2)
    plt.title('Focal Loss (Error Rate)', fontsize=14)
    plt.xlabel('Epochs')
    plt.ylabel('Loss Value')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()

    plt.tight_layout()
    plt.show()

# Execute the plot
plot_training_results(history)

In [ ]:
best_auc = max(history.history['val_auc_performance'])
best_acc = max(history.history['val_accuracy'])

print(f"--- FINAL PERFORMANCE REPORT ---")
print(f"Maximum Validation AUC: {best_auc:.2%}")
print(f"Maximum Validation Accuracy: {best_acc:.2%}")
print(f"Status: Target (95%) Achieved!" if best_auc > 0.95 else "Status: Optimization needed.")

In [ ]:
import pandas as pd
history_df = pd.DataFrame(history.history)
history_df.to_csv('training_history.csv', index=False)
from google.colab import files
files.download('training_history.csv')